# Train YOLO บน Colab (ดอกไม้ คลาสเดียว) — รอบ v3

1. **Runtime → Change runtime type → T4 GPU**
2. ใส่ Roboflow key: แถบซ้ายรูป 🔑 (Secrets) → secret ชื่อ `ROBOFLOW_API_KEY` → เปิด **Notebook access**
   - **v3 ต้องใช้ key ของ workspace `flowerworkspace`** (Roboflow → Settings → API Keys ของ workspace นั้น) เพราะ key เดิม (watash) มองไม่เห็นโปรเจค private `flower-count`
3. อัปโหลดโมเดลตัวเก่า `runs/detect/flower_single_yolo11/weights/best.pt` (ในเครื่อง) ไปไว้ใน Drive ที่ `MyDrive/flower_yolo/base_old.pt`
4. รันเซลล์ด้านล่าง (ครั้งแรกจะถามสิทธิ์ Google Drive → อนุญาต)

- ผลบันทึกลง Drive ทุก epoch → **ถ้าหลุด/ปิดแท็บ แค่รันเซลล์ซ้ำ จะ train ต่อจากเดิม**
- **v3:** ต่อยอดจากตัวเก่า (ดีสุดเรื่องนับดอกในช่อ) + Bouquet + รูปช่อที่ตีกล่องเอง (`flower-count`) | ไม่ใช้ 2 ชุดของ v2 (รูปดอกเดี่ยว ทำให้นับดอกในช่อแย่ลง)
- เสร็จแล้วดาวน์โหลด `MyDrive/flower_yolo/flower_single_v3/weights/best.pt`
  มาวางในเครื่องที่ `runs/detect/flower_single_v3/weights/best.pt`


In [ ]:
%pip install -q ultralytics roboflow
import os
from google.colab import drive, userdata
from roboflow import Roboflow
from ultralytics import YOLO

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/flower_yolo"   # เก็บใน Drive: runtime หลุดแล้วผลไม่หาย
BASE = f"{OUT}/base_old.pt"                  # ตัวเก่า (runs/detect/flower_single_yolo11) อัปโหลดเอง
RUN = "flower_single_v3"
last = f"{OUT}/{RUN}/weights/last.pt"

# (workspace, project, version)
# v2 ใช้ flower-classification-gsskn + flower-yvqgq แล้วนับดอกในช่อแย่ลง (ส่วนใหญ่เป็นรูปดอกเดี่ยว) → v3 ไม่ใช้
SETS = [("elbrusds-lzhr7", "bouquet", 8),           # ช่อดอกไม้ 1,318 รูป (public)
        ("flowerworkspace", "flower-count", 1)]    # รูปช่อที่ตีกล่องเอง 296 รูป (private: ต้องใช้ key ของ flowerworkspace)
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
# โหลดทุกครั้ง (runtime ใหม่ไม่มีไฟล์เดิม แม้แต่ตอน resume ก็ต้องใช้)
locs = [rf.workspace(w).project(p).version(v).download("yolov11", location=f"/content/data/{p}").location for w, p, v in SETS]
# data.yaml ของ Roboflow ใช้ path ../train ที่ ultralytics หาไม่เจอ → เขียน yaml รวมทุกชุดแบบ path เต็ม
# ชุดที่ไม่มี valid ข้ามไป | single_cls: เลขคลาสต่างกันในแต่ละชุดไม่เป็นไร
yaml = "/content/data/data_local.yaml"
with open(yaml, "w") as f:
    f.write("train: [" + ", ".join(f"'{l}/train/images'" for l in locs) + "]\n"
            "val: [" + ", ".join(f"'{l}/valid/images'" for l in locs if os.path.isdir(f"{l}/valid/images")) + "]\n"
            "nc: 1\nnames: ['flower']\n")

if os.path.exists(last):
    print("พบ last.pt ใน Drive → train ต่อจากเดิม")
    YOLO(last).train(resume=True)
else:
    # imgsz 640 เท่าตัวเก่า (ลอง 960 แล้วไม่ช่วย) | ต่อยอดจากตัวเก่า ใช้ epoch น้อย
    YOLO(BASE).train(data=yaml, epochs=60, imgsz=640, batch=16, patience=20,
                     single_cls=True, project=OUT, name=RUN, exist_ok=True)
print("เสร็จ! best.pt อยู่ที่", f"{OUT}/{RUN}/weights/best.pt")